# 🩺 Detecção de Neoplasias Mamárias com CNN + Random Forest

**TCC — MBA em Data Science & Analytics | USP/ESALQ 2025**  
**Autor:** André Luiz Magalhães de Oliveira  

---

## Objetivo

Classificar neoplasias mamárias em imagens de mamografia como **benignas (classe 0)** ou **malignas (classe 1)**, utilizando features extraídas pela rede neural **ResNet50** e classificadas com um pipeline **SMOTE + PCA + Random Forest**.

Este notebook implementa o **modelo de melhor desempenho** identificado no TCC, com AUC-ROC de 0.77 e acurácia de 70%.

## Dataset

**CBIS-DDSM** — Curated Breast Imaging Subset of DDSM  
Fonte: [The Cancer Imaging Archive (TCIA)](https://www.cancerimagingarchive.net/collection/cbis-ddsm/)

As features de entrada (`X`) são extraídas pela ResNet50 a partir das imagens de mamografia. O vetor `y` contém os rótulos binários (0 = sem neoplasia, 1 = neoplasia).

## Pipeline

```
Features ResNet50 → SMOTE → PCA → Random Forest → Classificação
```

| Etapa | Técnica | Objetivo |
|---|---|---|
| Balanceamento | SMOTE | Corrigir desbalanceamento de classes |
| Redução dimensional | PCA | Remover ruído de bordas e texturas |
| Classificação | Random Forest | Reduzir overfitting com ensemble de árvores |

## 1. Importações e Configuração

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from imblearn.over_sampling import SMOTE

# Configuração de logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

## 2. Funções Auxiliares

O código é organizado em funções reutilizáveis para facilitar experimentação com diferentes configurações de hiperparâmetros.

In [ ]:
def build_pipeline(random_state=42):
    """
    Cria o pipeline com SMOTE, PCA e Random Forest.
    
    O SMOTE é aplicado apenas nos dados de treino (dentro do pipeline),
    evitando vazamento de dados para o conjunto de teste.
    """
    logger.debug("Construindo pipeline com SMOTE, PCA e RandomForest.")
    return Pipeline([
        ('smote', SMOTE(random_state=random_state)),
        ('pca', PCA()),
        ('clf', RandomForestClassifier(random_state=random_state))
    ])


def get_param_grid():
    """
    Define o grid de hiperparâmetros para o GridSearchCV.
    
    - pca__n_components: número de componentes principais a manter
    - clf__n_estimators: número de árvores na floresta
    - clf__max_depth: profundidade máxima de cada árvore (None = sem limite)
    """
    logger.debug("Definindo grid de hiperparâmetros.")
    return {
        'pca__n_components': [10, 20, 30, 40, 50],
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [None, 10, 15, 20],
    }


def run_grid_search(pipeline, param_grid, X_train, y_train):
    """
    Executa o GridSearchCV com validação cruzada de 5 folds.
    
    Otimiza o f1_macro para garantir bom desempenho em ambas as classes,
    especialmente na classe 1 (casos positivos de neoplasia).
    """
    logger.info("Iniciando GridSearchCV...")
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=5,
        n_jobs=-1,
        verbose=2
    )
    grid_search.fit(X_train, y_train)
    logger.info("GridSearchCV concluído.")
    return grid_search


def evaluate_model(grid_search, X_test, y_test):
    """Avalia o modelo com os melhores hiperparâmetros encontrados."""
    logger.info("Avaliando modelo com melhores parâmetros...")
    y_pred = grid_search.best_estimator_.predict(X_test)

    logger.info(f"Melhores parâmetros: {grid_search.best_params_}")
    logger.info("Relatório de classificação:\n" + classification_report(y_test, y_pred))
    logger.info("Matriz de confusão:\n" + str(confusion_matrix(y_test, y_pred)))

    return y_pred


def plot_confusion_matrix(y_test, y_pred, labels):
    """Plota a matriz de confusão com heatmap."""
    logger.debug("Gerando gráfico da matriz de confusão.")
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão — Random Forest + SMOTE + PCA")
    plt.tight_layout()
    plt.show()


def plot_roc_curve(grid_search, X_test, y_test):
    """Plota a curva ROC e exibe o valor de AUC."""
    y_proba = grid_search.best_estimator_.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'SMOTE + PCA + RF (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
    plt.xlabel("Taxa de Falsos Positivos")
    plt.ylabel("Taxa de Positivos Verdadeiros")
    plt.title("Curva ROC")
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()
    logger.info(f"AUC-ROC: {roc_auc:.4f}")


def plot_precision_recall_curve(grid_search, X_test, y_test):
    """Plota a curva Precision-Recall e exibe o valor de AP."""
    y_proba = grid_search.best_estimator_.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, color='steelblue', lw=2, label=f'SMOTE + PCA + RF (AP = {ap:.2f})')
    plt.xlabel("Recall (Sensibilidade)")
    plt.ylabel("Precision (Precisão)")
    plt.title("Curva Precision-Recall")
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    logger.info(f"Average Precision (AP): {ap:.4f}")

## 3. Fluxo Principal

A função `main()` orquestra todo o pipeline:
1. Divide os dados (75% treino / 25% teste, estratificado)
2. Constrói e treina o pipeline com GridSearchCV
3. Avalia e visualiza os resultados

> **Nota:** `X` deve conter as features extraídas pela ResNet50 e `y` os rótulos binários (0 ou 1).

In [ ]:
def main(X, y):
    logger.info("Iniciando fluxo principal...")

    # Split estratificado para manter proporção de classes
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )
    logger.info(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")

    # Pipeline e grid de hiperparâmetros
    pipeline  = build_pipeline()
    param_grid = get_param_grid()

    # Treinamento com busca de melhores hiperparâmetros
    grid_search = run_grid_search(pipeline, param_grid, X_train, y_train)

    # Avaliação
    y_pred = evaluate_model(grid_search, X_test, y_test)

    # Visualizações
    plot_confusion_matrix(y_test, y_pred, labels=np.unique(y))
    plot_roc_curve(grid_search, X_test, y_test)
    plot_precision_recall_curve(grid_search, X_test, y_test)

    logger.info("Fluxo principal concluído.")


# Executa o fluxo principal
# Substitua X e y pelas features extraídas da ResNet50 e pelos rótulos do dataset
if __name__ == "__main__":
    main(X, y)

## 4. Resultados Esperados

Com base nos experimentos do TCC, os resultados obtidos com este modelo foram:

| Métrica | Classe 0 (negativo) | Classe 1 (positivo) |
|---|---|---|
| Acurácia | 70% | 70% |
| Precisão | 75% | 61% |
| Recall | 70% | 67% |
| AUC-ROC | — | **0.77** |
| AP (Precision-Recall) | — | **0.67** |

> Este foi o modelo com **melhor equilíbrio geral** entre os três avaliados no TCC, classificado como "Bom" segundo a escala AUC para aplicações clínicas.

## 5. Referência

Oliveira, A. L. M.; Bampi, H. *Utilização de redes neurais convolucionais para identificação de neoplasias mamárias.* TCC — Especialização em Data Science & Analytics, USP/ESALQ, 2025.